# xView2 Disaster Dataset Processing

This notebook processes the xView2 disaster dataset to analyze building damage patterns.

## Dataset Structure
```
../input/
├── images/
│   ├── disaster_name_pre_*.png
│   └── disaster_name_post_*.png
└── labels/
    └── disaster_name_post_*.json
```

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import json
from collections import Counter
from collections import defaultdict

In [ ]:
import json
from PIL import Image, ImageDraw
from IPython.display import display
from shapely import wkt
from shapely.geometry.multipolygon import MultiPolygon

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from copy import copy

%matplotlib inline
plt.rcParams['figure.figsize'] = 15, 15

In [ ]:
# Color codes for damage types
damage_dict = {
    "no-damage": (0, 255, 0, 50),
    "minor-damage": (0, 0, 255, 50),
    "major-damage": (255, 69, 0, 50),
    "destroyed": (255, 0, 0, 50),
    "un-classified": (255, 255, 255, 50)
}

In [ ]:
def get_damage_type(properties):
    """Extract damage type from polygon properties"""
    if 'subtype' in properties:
        return properties['subtype']
    else:
        return 'no-damage'

In [ ]:
def read_label(label_path):
    """Read JSON label file"""
    with open(label_path) as json_file:
        image_json = json.load(json_file)
        return image_json

In [ ]:
def annotate_img(draw, coords):
    """Draw damage polygons on image"""
    wkt_polygons = []

    for coord in coords:
        damage = get_damage_type(coord['properties'])
        wkt_polygons.append((damage, coord['wkt']))

    polygons = []

    for damage, swkt in wkt_polygons:
        polygons.append((damage, wkt.loads(swkt)))

    for damage, polygon in polygons:
        x, y = polygon.exterior.coords.xy
        coords = list(zip(x, y))
        draw.polygon(coords, damage_dict[damage])

In [ ]:
def display_img(json_path, time='post', annotated=True):
    """Display image with optional annotation"""
    json_path = str(json_path)
    if time == 'pre':
        json_path = json_path.replace('post', 'pre')

    img_path = json_path.replace('labels', 'images').replace('json', 'png')

    image_json = read_label(json_path)
    img_name = image_json['metadata']['img_name']

    print(f"Image: {img_name}")

    img = Image.open(img_path)
    draw = ImageDraw.Draw(img, 'RGBA')

    if annotated:
        annotate_img(draw, image_json['features']['xy'])

    return img

In [ ]:
def get_centroid(coords):
    polygons = [ wkt.loads(polygon['wkt']) for polygon in coords ]
    centroid =  MultiPolygon(polygons).centroid
    try:
        return {'centroid_x': centroid.x, 'centroid_y': centroid.y, 'latlong': centroid }
    except (IndexError, Exception) as e:
        return {'centroid_x': None, 'centroid_y': None, 'latlong': None }

In [ ]:
def get_damage_dict(coords):
    """Count damage types"""
    damage_list = [get_damage_type(coord['properties']) for coord in coords]
    return Counter(damage_list)

In [ ]:
def metadata_with_damage(label_path):
    """Extract metadata with damage information"""
    data = read_label(label_path)
    coords = data['features']['lng_lat']

    damage_dict = get_damage_dict(coords)
    centroid = get_centroid(coords)

    data['metadata'].update(centroid)
    data['metadata']['path'] = label_path
    data['metadata'].update(damage_dict)
    return data['metadata']

In [ ]:
def generate_metadata_df(labels_list):
    """Generate DataFrame from labels"""
    metadata_list = [metadata_with_damage(label_path) for label_path in labels_list]
    df = pd.DataFrame(metadata_list)
    # Fill NA with mean for numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    return df

## Load Dataset

Fetch all post-disaster JSON files from the dataset

In [ ]:
# Fetch all post disaster json files
labels_generator = Path('../input').rglob(pattern='*post_*.json')
labels_list = list(labels_generator)

print(f"Found {len(labels_list)} label files")

In [ ]:
# Group json files by disaster type
def get_disaster_dict(labels_list):
    disaster_dict = defaultdict(list)
    for label in labels_list:
        disaster_type = label.name.split('_')[0]
        disaster_dict[disaster_type].append(str(label))
    return disaster_dict

disaster_dict = get_disaster_dict(labels_list)

print("Disaster types in dataset:")
for disaster, files in disaster_dict.items():
    print(f"  {disaster}: {len(files)} files")

## Generate Metadata DataFrame

Process all labels and create a structured DataFrame with damage statistics

In [ ]:
# Generate metadata DataFrame for all labels
df = generate_metadata_df(labels_list)

print(f"DataFrame shape: {df.shape}")
print(f"\nDataFrame columns:\n{df.columns.tolist()}")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Sort by most damaged locations
sorted_df = df.sort_values(by=['destroyed'], ascending=False)
sorted_df[['disaster', 'img_name', 'destroyed', 'major-damage', 'minor-damage', 'no-damage']].head(10)

## Data Statistics

Analyze damage distribution across the dataset

In [ ]:
# Overall damage statistics
damage_cols = ['no-damage', 'minor-damage', 'major-damage', 'destroyed', 'un-classified']
damage_stats = df[damage_cols].sum()
print("Total damage counts:")
print(damage_stats)

In [ ]:
# Damage distribution by disaster type
disaster_damage = df.groupby('disaster')[damage_cols].sum()
disaster_damage

In [ ]:
# Plot damage distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Overall damage distribution
damage_stats.plot(kind='bar', ax=axes[0], color=['green', 'blue', 'orange', 'red', 'gray'])
axes[0].set_title('Overall Damage Distribution', fontsize=16)
axes[0].set_xlabel('Damage Type', fontsize=14)
axes[0].set_ylabel('Building Count', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)

# Damage by disaster type
disaster_damage.plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title('Damage Distribution by Disaster Type', fontsize=16)
axes[1].set_xlabel('Disaster Type', fontsize=14)
axes[1].set_ylabel('Building Count', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Damage Type', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('/kaggle/working/damage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Sample Image Visualization

Display pre/post disaster images with annotations

In [ ]:
# Select a sample image with most damage
sample_row = sorted_df.iloc[0]
label_path = sample_row['path']

print(f"Sample: {sample_row['disaster']}")
print(f"Image: {sample_row['img_name']}")
print(f"Destroyed: {int(sample_row['destroyed'])}, Major damage: {int(sample_row['major-damage'])}")

In [ ]:
# Display post-disaster image with annotations
img_post = display_img(label_path, time='post', annotated=True)
plt.imshow(img_post)
plt.title('Post-Disaster (Annotated)')
plt.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_post_disaster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Display pre-disaster image
img_pre = display_img(label_path, time='pre', annotated=False)
plt.imshow(img_pre)
plt.title('Pre-Disaster')
plt.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_pre_disaster.png', dpi=150, bbox_inches='tight')
plt.show()

## Export Processed Data

Save the processed metadata for further analysis

In [ ]:
# Save metadata DataFrame
output_path = '/kaggle/working/processed_metadata.csv'
df.to_csv(output_path, index=False)
print(f"Metadata saved to: {output_path}")
print(f"Total records: {len(df)}")

In [ ]:
# Save disaster-specific DataFrames
output_dir = '/kaggle/working/disaster_data'
os.makedirs(output_dir, exist_ok=True)

for disaster, labels in disaster_dict.items():
    disaster_df = generate_metadata_df(labels)
    disaster_path = os.path.join(output_dir, f'{disaster}_metadata.csv')
    disaster_df.to_csv(disaster_path, index=False)
    print(f"Saved {disaster}: {len(disaster_df)} records")

## Verification

Verify the processed data format and structure

In [ ]:
# Load and verify the exported data
verify_df = pd.read_csv('/kaggle/working/processed_metadata.csv')

print("=== Data Verification ===")
print(f"Shape: {verify_df.shape}")
print(f"\nData types:\n{verify_df.dtypes}")
print(f"\nMissing values:\n{verify_df.isnull().sum()}")
print(f"\nFirst few rows:")
verify_df.head()

In [ ]:
# Check file outputs
import os

print("=== Output Files ===")
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        file_path = os.path.join(root, file)
        file_size = os.path.getsize(file_path)
        print(f"{file_path} ({file_size} bytes)")

## Summary

This notebook has processed the xView2 disaster dataset and:
1. Loaded all post-disaster JSON label files
2. Extracted metadata including damage classifications
3. Calculated centroids for polygon groups
4. Created a structured DataFrame with damage statistics
5. Generated visualizations of damage distribution
6. Exported processed data in CSV format

The processed data can now be used for:
- Training machine learning models
- Further statistical analysis
- Visualization and reporting